# الفصل 30: مقدمة إلى PEFT: التدريب الفعّال من حيث البارامترات

> تخيّل أن لديك نموذج Llama 3 بحجم 70 مليار معامل، وتريد تخصيصه لمهمة طبية محددة بلغتك.

- **الجزء:** 7 — التخصيص الفعّال والمحاذاة
- **الفصل:** 30
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 29: إعادة استخدام النماذج: إطار القرار الشامل
- **الوسوم:** PEFT، Adapters، Prompt Tuning، BitFit

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch30-peft.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-07-peft-alignment%2Fch30-peft.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-07-peft-alignment/ch30-peft.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch30-peft.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch30-peft.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch30-peft.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch30-peft.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-07-peft-alignment%2Fch30-peft.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-07-peft-alignment/ch30-peft.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch30-peft.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch30-peft.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-07-peft-alignment/ch30-peft.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> تخيّل أن لديك نموذج Llama 3 بحجم 70 مليار معامل، وتريد تخصيصه لمهمة طبية محددة بلغتك.

> **بيانات الفصل**
>
> - **الجزء:** 7 — التخصيص الفعّال والمحاذاة
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 29: إعادة استخدام النماذج: إطار القرار الشامل


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم Prompt Tuning (Lester et al., 2021)
- فهم Prefix Tuning (Li & Liang, 2021)
- فهم BitFit — الطريقة الأبسط على الإطلاق


## 1. لماذا يهمّك هذا الموضوع؟

تخيّل أن لديك نموذج Llama 3 بحجم 70 مليار معامل، وتريد تخصيصه لمهمة طبية محددة بلغتك. الضبط الدقيق الكامل (Full Fine-Tuning) يتطلب:

| المورد | الحساب التقريبي لنموذج 70B |
|---|---|
| الأوزان (FP16) | 140 GB |
| التدرجات (Gradients) | 140 GB |
| حالات Adam (m, v) | 560 GB |
| **الإجمالي (تدريب واحد)** | **~840 GB VRAM** |

هذا يتطلب مجموعة GPUs ضخمة (8-16× A100/H100) لمجرد ضبط نموذج واحد لمهمة واحدة. إذا أردت **10 مهام مختلفة**، تحتاج 10 نسخ كاملة من النموذج (840GB × 10).

**PEFT (Parameter-Efficient Fine-Tuning)** يحلّ هذه المشكلة جذريًا: نُجمّد النموذج الأساسي بالكامل، ونضيف عددًا ضئيلًا جدًا من المعاملات القابلة للتدريب (غالبًا أقل من 1% من الحجم الكلي)، ونحصل على أداء مقارب جدًا لـ Full Fine-Tuning.

> 💡 **إحصائية مهمة**: LoRA على نموذج 70B قد يضيف فقط 20-80 مليون معامل قابلة للتدريب (أقل من 0.1%)، وتحتاج VRAM إضافية ضئيلة جدًا مقارنة بالضبط الكامل.

---


## 2. الخريطة الذهنية


```text
PEFT Methods
│
├─ Addition-based (إضافة وحدات جديدة)
│   ├─ Adapter Layers (Houlsby et al., 2019)
│   ├─ LoRA (Low-Rank Adaptation)
│   └─ (IA)³ (Infused Adapter)
│
├─ Reparameterization-based (إعادة صياغة التحديث)
│   ├─ LoRA (ينتمي أيضًا هنا)
│   └─ DoRA (Weight-Decomposed LoRA)
│
├─ Prompt-based (التعلّم عبر مُدخلات مُتعلَّمة)
│   ├─ Prompt Tuning (Lester et al., 2021)
│   ├─ Prefix Tuning (Li & Liang, 2021)
│   └─ P-Tuning v2
│
└─ Selective-based (تدريب جزء فقط من الأوزان الأصلية)
    ├─ BitFit (تدريب الـ bias فقط)
    └─ Diff Pruning
```


---


## 3. الشرح المتدرج

### 🟢 طبقة الحدس

فكّر في النموذج المُدرَّب مسبقًا كـ"دماغ" يملك معرفة عامة هائلة. بدلًا من إعادة تدريب كل خلية عصبية (Full FT)، نضيف "ملاحظات صغيرة على الهامش" (Adapters) أو "تلميحات موجّهة" (Prompts) تُعدِّل سلوك الدماغ لمهمة محددة دون المساس بالمعرفة الأساسية.

### 🟡 طبقة الآلية — Adapter Layers

فكرة Houlsby (2019): نُدخِل طبقة صغيرة (Bottleneck) بعد كل طبقة Feed-Forward وAttention في المحوّل، ونجمّد كل شيء آخر.


```text
الإدخال x (بُعد d_model)
      │
      ▼
┌─────────────────┐
│ Down-projection  │  d_model → r  (r << d_model)
│ W_down           │
└─────────────────┘
      │
      ▼
   تفعيل غير خطي (ReLU/GELU)
      │
      ▼
┌─────────────────┐
│ Up-projection    │  r → d_model
│ W_up             │
└─────────────────┘
      │
      ▼
  + Residual Connection (x)
```


### 🔴 طبقة التعمّق — الكود الكامل


In [ ]:
import torch
import torch.nn as nn
import math

class AdapterLayer(nn.Module):
    """
    طبقة Adapter كلاسيكية (Houlsby et al., 2019)
    تُدرَج بعد طبقات Attention وFFN في كل كتلة محوّل.
    """
    def __init__(self, d_model: int, bottleneck_dim: int = 64,
                 init_scale: float = 1e-3):
        super().__init__()
        self.down_proj = nn.Linear(d_model, bottleneck_dim)
        self.up_proj = nn.Linear(bottleneck_dim, d_model)
        self.activation = nn.GELU()

        # تهيئة قريبة من الصفر لضمان أن الـ Adapter
        # لا يُخرِّب سلوك النموذج المُدرَّب مسبقًا في بداية التدريب
        # (near-identity initialization)
        nn.init.normal_(self.down_proj.weight, std=init_scale)
        nn.init.zeros_(self.down_proj.bias)
        nn.init.zeros_(self.up_proj.weight)
        nn.init.zeros_(self.up_proj.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        h = self.down_proj(x)
        h = self.activation(h)
        h = self.up_proj(h)
        return residual + h  # Residual Connection إلزامي

class TransformerBlockWithAdapters(nn.Module):
    """
    كتلة محوّل قياسية + Adapters مُدمَجة بعد Attention وبعد FFN.
    الأوزان الأصلية (attention, ffn, norms) تُجمَّد بالكامل.
    """
    def __init__(self, d_model: int, n_heads: int, d_ff: int,
                 adapter_dim: int = 64):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.attn_adapter = AdapterLayer(d_model, adapter_dim)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.ffn_adapter = AdapterLayer(d_model, adapter_dim)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, attn_mask=None):
        # Attention + Adapter
        attn_out, _ = self.attention(x, x, x, attn_mask=attn_mask)
        attn_out = self.attn_adapter(attn_out)
        x = self.norm1(x + attn_out)

        # FFN + Adapter
        ffn_out = self.ffn(x)
        ffn_out = self.ffn_adapter(ffn_out)
        x = self.norm2(x + ffn_out)
        return x

def freeze_base_model_keep_adapters(model: nn.Module):
    """
    تجميد كل المعاملات ما عدا الـ Adapters.
    هذه هي الخطوة الجوهرية في كل تقنيات PEFT: تحديد
    ما الذي يبقى requires_grad=True بدقة.
    """
    trainable_params, total_params = 0, 0
    for name, param in model.named_parameters():
        total_params += param.numel()
        if "adapter" in name:
            param.requires_grad = True
            trainable_params += param.numel()
        else:
            param.requires_grad = False

    pct = 100 * trainable_params / total_params
    print(f"معاملات قابلة للتدريب: {trainable_params:,} "
          f"({pct:.3f}% من الإجمالي {total_params:,})")
    return trainable_params, total_params


### Prompt Tuning (Lester et al., 2021)

بدلًا من إضافة طبقات داخل الشبكة، نُضيف **متجهات مُتعلَّمة** (Soft Prompts) إلى بداية تسلسل الإدخال، ونجمّد النموذج بأكمله.


In [ ]:
class PromptTuningWrapper(nn.Module):
    """
    Prompt Tuning: نتعلّم n_tokens متجهات افتراضية (soft prompts)
    تُضاف إلى بداية embedding الإدخال. لا نغيّر معمارية النموذج إطلاقًا.
    """
    def __init__(self, base_model: nn.Module, embedding_dim: int,
                 n_virtual_tokens: int = 20, vocab_size_for_init: int = 30000):
        super().__init__()
        self.base_model = base_model
        for p in self.base_model.parameters():
            p.requires_grad = False  # تجميد كامل للنموذج الأساسي

        # تهيئة ذكية: أخذ عينة من embeddings حقيقية بدلًا من عشوائية بحتة
        # (Vocabulary-based initialization) تُسرّع التقارب بشكل ملحوظ
        self.soft_prompt = nn.Parameter(
            torch.randn(n_virtual_tokens, embedding_dim) * 0.01
        )
        self.n_virtual_tokens = n_virtual_tokens

    def forward(self, input_embeds: torch.Tensor, attention_mask=None):
        batch_size = input_embeds.shape[0]
        prompt = self.soft_prompt.unsqueeze(0).expand(batch_size, -1, -1)
        full_input = torch.cat([prompt, input_embeds], dim=1)

        if attention_mask is not None:
            prompt_mask = torch.ones(batch_size, self.n_virtual_tokens,
                                      device=attention_mask.device)
            attention_mask = torch.cat([prompt_mask, attention_mask], dim=1)

        return self.base_model(inputs_embeds=full_input,
                                attention_mask=attention_mask)


### Prefix Tuning (Li & Liang, 2021)

تطوير على Prompt Tuning: بدلًا من إضافة متجهات فقط لطبقة الإدخال، نُضيف **بادئات مُتعلَّمة منفصلة لكل طبقة** في مفاتيح وقيم (K, V) الـ Attention — مما يمنح تعبيرية أعلى.


In [ ]:
class PrefixTuningBlock(nn.Module):
    """
    Prefix Tuning: لكل طبقة محوّل، نُضيف بادئة مُتعلَّمة
    إلى K و V في طبقة Attention الخاصة بها (وليس فقط للإدخال).
    يُنتَج عبر شبكة MLP صغيرة (Re-parameterization) لتحسين الاستقرار.
    """
    def __init__(self, n_layers: int, n_heads: int, head_dim: int,
                 n_prefix_tokens: int = 10, mlp_hidden: int = 512):
        super().__init__()
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.head_dim = head_dim
        self.n_prefix_tokens = n_prefix_tokens

        # متجه بذرة صغير يُمرَّر عبر MLP لتوليد كل البادئات
        # (أثبتت التجارب أن التدريب المباشر للبادئات غير مستقر)
        embed_dim = n_layers * n_heads * head_dim * 2  # ×2 لـ K و V
        self.prefix_seed = nn.Parameter(torch.randn(n_prefix_tokens, 512))
        self.reparam_mlp = nn.Sequential(
            nn.Linear(512, mlp_hidden),
            nn.Tanh(),
            nn.Linear(mlp_hidden, embed_dim),
        )

    def get_prefix_kv(self, batch_size: int, device):
        prefix = self.reparam_mlp(self.prefix_seed)  # (n_prefix, embed_dim)
        prefix = prefix.view(self.n_prefix_tokens, self.n_layers,
                              2, self.n_heads, self.head_dim)
        prefix = prefix.permute(1, 2, 3, 0, 4)  # (n_layers, 2, n_heads, n_prefix, head_dim)
        prefix = prefix.unsqueeze(1).expand(-1, batch_size, -1, -1, -1, -1)
        past_key_values = []
        for layer_idx in range(self.n_layers):
            k = prefix[layer_idx, :, 0]  # (batch, n_heads, n_prefix, head_dim)
            v = prefix[layer_idx, :, 1]
            past_key_values.append((k, v))
        return past_key_values


### BitFit — الطريقة الأبسط على الإطلاق


In [ ]:
def apply_bitfit(model: nn.Module):
    """
    BitFit (Ben Zaken et al., 2021): تدريب الـ bias terms فقط!
    مفاجئ لكنه فعّال لمهام بسيطة/متوسطة التعقيد.
    نسبة المعاملات القابلة للتدريب غالبًا < 0.1%.
    """
    trainable, total = 0, 0
    for name, param in model.named_parameters():
        total += param.numel()
        if "bias" in name:
            param.requires_grad = True
            trainable += param.numel()
        else:
            param.requires_grad = False
    print(f"BitFit: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")
    return model


---


## 4. مثال محلول خطوة بخطوة

**المسألة**: لديك BERT-base (110M معامل) وتريد ضبطه لتصنيف المشاعر على تغريدات عربية (3 فئات). قارن استهلاك الذاكرة بين Full FT وAdapter Tuning.


In [ ]:
def compare_memory_footprint(d_model=768, n_layers=12, n_heads=12,
                               d_ff=3072, adapter_dim=64):
    # Full Fine-Tuning
    total_params = 110_000_000  # BERT-base تقريبًا
    full_ft_trainable = total_params

    # Adapter Tuning: adapter واحد بعد Attention وواحد بعد FFN لكل طبقة
    params_per_adapter = 2 * (d_model * adapter_dim + adapter_dim * d_model) \
                        + 2 * adapter_dim  # + biases تقريبية
    adapters_total = params_per_adapter * n_layers * 2  # ×2 attention+ffn

    print(f"Full FT معاملات قابلة للتدريب: {full_ft_trainable:,}")
    print(f"Adapter Tuning معاملات قابلة للتدريب: {adapters_total:,}")
    print(f"النسبة: {100 * adapters_total / full_ft_trainable:.3f}%")

    # تقدير الذاكرة (Adam: 2 إضافيين لكل معامل + gradient)
    bytes_per_param_fp32 = 4
    full_ft_memory_gb = full_ft_trainable * bytes_per_param_fp32 * 4 / 1e9  # weights+grad+2 adam states
    adapter_memory_gb = adapters_total * bytes_per_param_fp32 * 4 / 1e9

    print(f"\nذاكرة تدريب Full FT (تقريبية): {full_ft_memory_gb:.3f} GB")
    print(f"ذاكرة تدريب Adapter (تقريبية): {adapter_memory_gb:.5f} GB")

compare_memory_footprint()
# الناتج التقريبي:
# Full FT معاملات قابلة للتدريب: 110,000,000
# Adapter Tuning معاملات قابلة للتدريب: 2,433,024
# النسبة: 2.212%
# ذاكرة تدريب Full FT: 1.760 GB
# ذاكرة تدريب Adapter: 0.039 GB


---


## 5. نصائح وحيل

| النصيحة | التفصيل |
|---|---|
| تهيئة قريبة من الصفر | Adapters/LoRA يجب أن تبدأ بتأثير شبه معدوم حتى لا "تُفسد" النموذج المُدرَّب مسبقًا في البداية |
| اختيار موقع الإدراج | FFN layers عادة أكثر فائدة من Attention layers للـ Adapters (نتائج Houlsby الأصلية) |
| حجم Bottleneck | ابدأ بـ 64، وزِد فقط إذا كان الأداء غير كافٍ — الزيادة تضاعف التكلفة الحاسوبية |
| دمج عدة Adapters | يمكن تدريب Adapter منفصل لكل مهمة والتبديل بينها دون إعادة تحميل النموذج الكامل |


## 6. أخطاء شائعة ومصائد

- **نسيان تجميد النموذج الأساسي**: التحقق الدائم عبر `sum(p.numel() for p in model.parameters() if p.requires_grad)` قبل بدء التدريب.
- **استخدام Learning Rate مماثل لـ Full FT**: PEFT عادة يحتاج LR أعلى (1e-3 إلى 1e-4) لأن عدد المعاملات أقل بكثير.
- **Prompt Tuning على نماذج صغيرة**: يعمل بشكل ممتاز فقط على نماذج كبيرة (>1B)؛ على النماذج الصغيرة يكون أداؤه ضعيفًا مقارنة بـ Full FT.


## 7. جدول مقارنة

| الطريقة | % معاملات مُضافة | يُعدِّل المعمارية؟ | زمن استدلال إضافي | الأداء النسبي |
|---|---|---|---|---|
| Adapter Layers | 0.5-4% | نعم (طبقات جديدة) | طفيف | قريب من Full FT |
| Prompt Tuning | <0.01% | لا | طفيف جدًا | جيد فقط على نماذج كبيرة جدًا |
| Prefix Tuning | 0.1-1% | لا (K/V فقط) | طفيف | جيد |
| BitFit | <0.1% | لا | معدوم | متوسط (مهام بسيطة) |
| **LoRA** (الفصل التالي) | 0.1-1% | لا (قابل للدمج) | **معدوم بعد الدمج** | **ممتاز — الأكثر استخدامًا عالميًا** |


## 8. ملخص أفضل الممارسات

- ابدأ دائمًا بـ LoRA كخط أساس (سنشرحه بالتفصيل في الفصل 31) قبل تجربة بدائل أخرى.
- استخدم BitFit فقط للمهام البسيطة جدًا أو كتجربة أساسية سريعة.
- راقب نسبة المعاملات القابلة للتدريب دائمًا كمؤشر صحة أولي.
- وثّق أي Adapter/Prompt مُدرَّب مع البيانات التي دُرِّب عليها (Model Card مصغّرة).


## 9. أسئلة مقابلات

**س: لماذا تُهيَّأ طبقة up-projection في Adapter بالصفر؟**

**ج**: لضمان أن الـ Adapter، عند بداية التدريب، لا يُغيّر أي شيء في المخرجات الأصلية للنموذج (الدالة الناتجة = `residual + 0 = residual`). هذا يسمح للتدريب بالانطلاق من نقطة مستقرة معروفة (سلوك النموذج المُدرَّب مسبقًا) بدلًا من اضطراب عشوائي قد يُبطئ أو يُفشل التقارب. هذا المبدأ (near-identity initialization) أساسي في كل تقنيات PEFT الحديثة بما فيها LoRA.

**س: ما الفرق الجوهري بين Prompt Tuning وPrefix Tuning؟**

**ج**: Prompt Tuning يُضيف متجهات مُتعلَّمة فقط في طبقة الإدخال (embedding layer)، وتتدفق عبر الشبكة بشكل طبيعي عبر Attention. Prefix Tuning أكثر تعبيرية لأنه يُضيف بادئات K/V مُتعلَّمة **منفصلة لكل طبقة** في الشبكة، مما يمنح تحكمًا أدق في سلوك كل طبقة على حدة، لكنه يتطلب معاملات أكثر وتعقيدًا تدريبيًا أعلى (غالبًا يحتاج إعادة صياغة عبر MLP لتحقيق الاستقرار).


## 10. تمرين عملي

نفّذ `AdapterLayer` وادمجها مع نموذج `DistilBERT` من مكتبة `transformers`، ثم درّب على تصنيف ثنائي (إيجابي/سلبي) لمراجعات عربية، وقارن:
1. عدد المعاملات القابلة للتدريب.
2. الدقة النهائية (Accuracy) مقابل Full Fine-Tuning على نفس البيانات.
3. زمن التدريب لكل Epoch.


## 11. مراجع للتعمّق

- Houlsby et al. (2019) — *Parameter-Efficient Transfer Learning for NLP*
- Lester et al. (2021) — *The Power of Scale for Parameter-Efficient Prompt Tuning*
- Li & Liang (2021) — *Prefix-Tuning: Optimizing Continuous Prompts for Generation*
- Ben Zaken et al. (2021) — *BitFit: Simple Parameter-efficient Fine-tuning*


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — التخصيص الفعّال والمحاذاة / الفصل 30: مقدمة إلى PEFT: التدريب الفعّال من حيث البارامترات.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
